In [1]:
%pwd


'/Users/wft08/Desktop/CHATBOTAI 2/medibot/research'

In [2]:
import os
os.chdir("../")


In [3]:

%pwd


'/Users/wft08/Desktop/CHATBOTAI 2/medibot'

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents
extracted_data=load_pdf_file(data="/Users/wft08/Desktop/CHATBOTAI 2/Data")
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 12


In [5]:
extracted_data

[Document(metadata={'producer': 'macOS Version 14.6 (Build 23G80) Quartz PDFContext', 'creator': 'Adobe Illustrator 26.5 (Macintosh)', 'creationdate': '2025-05-08T05:37:57+00:00', 'creatorversion': '21.0.0', 'moddate': '2025-06-05T10:47:05+05:30', 'title': 'foursight-challenge-navigator', 'source': '/Users/wft08/Desktop/CHATBOTAI 2/Data/foursight.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Clarify\nGoal\nChallenge\n What do you want to accomplish? Finish the sentence, “It would be great if...”\nNow pinpoint t\nhe right challenge question. What question, if answered, would lead to a breakthrough? Review your data \nuntil you ﬁnd one that frames the right problem. Invite new thinking by beginning each question with a phrase like:\nPut a  by the question that points you in the direction of a breakthrough. Write it at the top of the next page.\n1. How to...?\n2. How might...?\n3. In what ways might...?\n4. What might be all the...?\n5.\n6.\n7.\n8.\n9.\n10.\nIt woul

In [6]:
from langchain.embeddings import HuggingFaceEmbeddings

In [7]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [8]:
embeddings = download_hugging_face_embeddings()

/var/folders/1p/rxdlwgxj1lb9j3q3x30x2l4c0000gn/T/ipykernel_2409/4238859041.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/Users/wft08/Desktop/CHATBOTAI 2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

/Users/wft08/Desktop/CHATBOTAI 2/venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Length 384


In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
import os
PINECONE_API_KEY= os.environ.get("PINECONE_API_KEY")


In [12]:
from dotenv import load_dotenv
load_dotenv()

import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1: Load API key
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

# Step 2: Create Pinecone client instance (NO init)
pc = Pinecone(api_key=PINECONE_API_KEY)

# Step 3: Index name
index_name = "test"

# Step 4: (Optional) Create index if not exists
if index_name not in [index["name"] for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,  # Must match the embedding model's output dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# Step 5: Load PDF and split
loader = PyPDFLoader("/Users/wft08/Desktop/CHATBOTAI 2/Data/foursight.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
text_chunks = text_splitter.split_documents(documents)

# Step 6: Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 7: Store in Pinecone
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)


/Users/wft08/Desktop/CHATBOTAI 2/venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [13]:
from pinecone import Pinecone

In [14]:
query = "How to build good habits?"
results = docsearch.similarity_search(query, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n{r.page_content}")



Result 1:
around,	build	a	life	around.
There	is	no	one	right	way	to	create	better	habits,	but	this	book	describes	the
best	way	I	know—an	approach	that	will	be	effective	regardless	of	where	you
start	or	what	you’re	trying	to	change.	The	strategies	I	cover	will	be	relevant	to
anyone	looking	for	a	step-by-step	system	for	improvement,	whether	your	goals
center	on	health,	money,	productivity,	relationships,	or	all	of	the	above.	As	long
as	human	behavior	is	involved,	this	book	will	be	your	guide.

Result 2:
around,	build	a	life	around.
There	is	no	one	right	way	to	create	better	habits,	but	this	book	describes	the
best	way	I	know—an	approach	that	will	be	effective	regardless	of	where	you
start	or	what	you’re	trying	to	change.	The	strategies	I	cover	will	be	relevant	to
anyone	looking	for	a	step-by-step	system	for	improvement,	whether	your	goals
center	on	health,	money,	productivity,	relationships,	or	all	of	the	above.	As	long
as	human	behavior	is	involved,	this	book	will	be	your	guide.

Resul

In [15]:
from dotenv import load_dotenv
load_dotenv()

import os
import pinecone
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers
from langchain_pinecone import PineconeVectorStore

llm = CTransformers(
    model="/Users/wft08/Desktop/CHATBOTAI 2/medibot/research/model/llama-2-7b-chat.ggmlv3.q4_0.bin",
    model_type="llama",
    config={
        "max_new_tokens": 56,           # Reduce token size to speed up
        "temperature": 0.3,
        "threads": 4,                    # Use 4 CPU threads if available
        "batch_size": 8,                 # Lower = less RAM needed
        "context_length": 2048          # Match model context (adjust if needed)
    }
)

print("Model loaded.")
print("Waiting for user input...")
print("Testing LLM response:")
response = llm("i am afraid to learn what should i do?")
print("Response test:", response)





Model loaded.
Waiting for user input...
Testing LLM response:


/var/folders/1p/rxdlwgxj1lb9j3q3x30x2l4c0000gn/T/ipykernel_2409/3505768248.py:29: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm("i am afraid to learn what should i do?")


Response test: 
 Unterscheidung between fear and anxiety 
Fear is a normal human emotion that can help us protect ourselves from danger. Anxiety, on the other hand, is an excessive or irrational fear of something that is unlikely to happen. It can interf


In [16]:
print("🔍 Testing LLM response:")
response = llm("how doing 1 percent  extra each day works?")
print("Response test:", response)

🔍 Testing LLM response:
Response test: 
 Unterscheidung between the two strategies:

The 1% rule is a simple yet powerful strategy for achieving financial freedom. The idea is to save or invest 1% of your income each day, and by doing so, you will eventually accumulate enough wealth to


In [17]:
from langchain.prompts import PromptTemplate

template = """
You are a helpful assistant. Use the provided context to answer the question clearly and concisely in 1–2 sentences. Ensure your response ends with a complete sentence.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


In [18]:
from langchain.chains import RetrievalQA
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore.from_existing_index(
    index_name="medicalbot",
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)


In [19]:
query = "what is next js ?"
response = qa_chain.run(query)

print("Response:\n", response)


/var/folders/1p/rxdlwgxj1lb9j3q3x30x2l4c0000gn/T/ipykernel_2409/1354561008.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain.run(query)
/Users/wft08/Desktop/CHATBOTAI 2/venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Response:
 Next.js is a popular JavaScript framework for building server-side rendered (SSR) web applications. It provides a set of features and tools to simplify the development process, including automatic code splitting, efficient module management, and optimized performance.


In [20]:
!which python
!pip list | grep huggingface


/Users/wft08/Desktop/CHATBOTAI 2/venv/bin/python


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface-hub           0.34.3
langchain-huggingface     0.3.1


In [21]:
%pip install ipywidgets


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached ipywidgets-8.1.7-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.14-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.7-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl (216 kB)
Using cached widgetsnbextension-4.0.14-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]
Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install ipywidgets jupyterlab_widgets


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [23]:
import ipywidgets as widgets
from IPython.display import display

# UI Elements
chat_output = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'overflow_y': 'auto'})
user_input = widgets.Text(placeholder='Ask something and press Enter...')

# Display the UI
display(chat_output, user_input)

# Clear previous bindings if this cell is re-run
try:
    user_input._submission_callbacks.callbacks.clear()
except:
    pass

# Handler
def on_enter(sender):
    question = user_input.value.strip()
    if not question:
        return
    if question.lower() in ['exit', 'quit']:
        with chat_output:
            print("🔚 Chatbot ended.")
        user_input.disabled = True
        return

    with chat_output:
        print(f"🧑 You: {question}")
        try:
            answer = qa_chain.run(question)
        except Exception as e:
            answer = f"[Error] {str(e)}"
        print(f"🤖 Bot: {answer}\n")

    user_input.value = ''  # Clear the input

# Bind event
user_input.on_submit(on_enter)


Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

Text(value='', placeholder='Ask something and press Enter...')

/var/folders/1p/rxdlwgxj1lb9j3q3x30x2l4c0000gn/T/ipykernel_2409/518077409.py:39: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  user_input.on_submit(on_enter)


In [24]:
%pip install fastapi "uvicorn[standard]" jinja2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached httptools-0.6.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (3.6 kB)
  Using cached uvloop-0.21.0-cp311-cp311-macosx_10_9_universal2.whl.metadata (4.9 kB)
  Using cached watchfiles-1.1.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached websockets-15.0.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.8 kB)
Using cached httptools-0.6.4-cp311-cp311-macosx_11_0_arm64.whl (103 kB)
Using cached uvloop-0.21.0-cp311-cp311-macosx_10_9_universal2.whl (1.4 MB)
Using cached watchfiles-1.1.0-cp311-cp311-macosx_11_0_arm64.whl (397 kB)
Using cached websockets-15.0.1-cp311-cp311-macosx_11_0_arm64.whl (173 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [watchfiles]
Note: you may need to restart the kernel to use updated packages.


In [25]:
from dotenv import load_dotenv
import os

load_dotenv()  # This loads .env file into environment variables

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_API_ENV = os.getenv("PINECONE_API_ENV")

print(f"Pinecone API Key: {PINECONE_API_KEY}")  # Debug: should print your key
print(f"Pinecone Env: {PINECONE_API_ENV}")      # Debug: should print your env


Pinecone API Key: pcsk_hqG9n_HyPe5ou2GF5QZgADWUpN8H3nKuZmPL3YMJAjFVsTASrfk8qX8kJNTHHAFxa9Nbc
Pinecone Env: us-east-1


In [26]:
!pip list

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Package                   Version     Editable project location
------------------------- ----------- ----------------------------------------
aiohappyeyeballs          2.6.1
aiohttp                   3.12.14
aiohttp-retry             2.9.1
aiosignal                 1.4.0
annotated-types           0.7.0
anyio                     4.9.0
appnope                   0.1.4
asttokens                 3.0.0
attrs                     25.3.0
blinker                   1.9.0
certifi                   2025.7.14
cffi                      1.17.1
charset-normalizer        3.4.2
click                     8.2.1
comm                      0.2.3
ctransformers             0.2.5
dataclasses-json          0.6.7
debugpy                   1.8.15
decorator                 5.2.1
distro                    1.9.0
executing                 2.2.0
fastapi                   0.116.1
filelock                  3.18.0
Flask                     3.1.1
frozenlist                1.7.0
fsspec                    2025.7.0
Generative